In [ ]:
import pandas as pd
import numpy as np
import zipfile
import requests

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import itertools, re

In [ ]:
from google.colab import files
files.upload()

Saving movies_clean.csv to movies_clean.csv
Saving kaggle.json to kaggle.json


{'movies_clean.csv': b'movieId,imdbId,tmdbId,id,title,vote_average,vote_count,release_date,revenue,runtime,adult,backdrop_path,budget,homepage,original_language,original_title,overview,popularity,poster_path,tagline,genres_y,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast,titleType\n1,114709,862.0,862,Toy Story,7.971,17152,1995-10-30,394400000,81,False,/3Rfvhy1Nl6sSGJwyjb0QiZzZYlB.jpg,30000000,http://toystory.disney.com/toy-story,en,Toy Story,"Led by Woody, Andy\'s toys live happily in his room until Andy\'s birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy\'s heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.",78.404,/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,Hang on for the comedy that goes to infinity and beyond!,"Animation, Adventure, Family, Comedy",Pixar,United States of America,English,"re

In [ ]:
import os

os.makedirs("/root/.kaggle", exist_ok=True)
!mv kaggle.json /root/.kaggle/

In [ ]:
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import itertools, re
import gc

print("=" * 60)
print("LOADING DATA...")
print("=" * 60)


df =  pd.read_csv("train_movies.csv")
df.head(20)

LOADING DATA...


,movieId,imdbId,tmdbId,id,title,vote_average,vote_count,release_date,revenue,runtime,...,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast,titleType
0,1,114709,862.0,862,Toy Story,7.971,17152,1995-10-30,394400000,81,...,Pixar,United States of America,English,"rescue, friendship, mission, martial arts, jea...",John Lasseter,"Joss Whedon, Andrew Stanton, Joel Cohen, Alec ...",8.3,1169713,"Tom Hanks, Tim Allen, Don Rickles, Jim Varney,...",movie
1,2,113497,8844.0,8844,Jumanji,7.239,9833,1995-12-15,262821940,104,...,"TriStar Pictures, Interscope Communications, T...",United States of America,"English, French","giant insect, board game, disappearance, jungl...",Joe Johnston,"Jonathan Hensleigh, Greg Taylor, Jim Strain, C...",7.1,411589,"Robin Williams, Kirsten Dunst, Bradley Pierce,...",movie
2,3,113228,15602.0,15602,Grumpier Old Men,6.476,347,1995-12-22,71500000,101,...,"Lancaster Gate, Warner Bros. Pictures",United States of America,English,"fishing, sequel, old man, best friend, wedding...",Howard Deutch,Mark Steven Johnson,6.7,31619,"Walter Matthau, Jack Lemmon, Ann-Margret, Soph...",movie
3,4,114885,31357.0,31357,Waiting to Exhale,6.183,142,1995-12-22,81452156,127,...,20th Century Fox,United States of America,English,"based on novel or book, interracial relationsh...",Forest Whitaker,"Terry McMillan, Ron Bass",6.0,13773,"Whitney Houston, Angela Bassett, Loretta Devin...",movie
4,5,113041,11862.0,11862,Father of the Bride Part II,6.228,659,1995-12-08,76594107,106,...,"Touchstone Pictures, Sandollar Productions",United States of America,English,"daughter, baby, parent child relationship, mid...",Charles Shyer,"Albert Hackett, Frances Goodrich, Nancy Meyers...",6.1,45035,"Steve Martin, Diane Keaton, Martin Short, Kimb...",movie
5,6,113277,949.0,949,Heat,7.903,6552,1995-12-15,187436818,170,...,"Regency Enterprises, Forward Pass, Warner Bros...",United States of America,"English, Spanish","robbery, chase, obsession, detective, heist, t...",Michael Mann,Michael Mann,8.3,799877,"Al Pacino, Robert De Niro, Val Kilmer, Jon Voi...",movie
6,7,114319,11860.0,11860,Sabrina,6.159,555,1995-12-15,53672080,127,...,"Worldwide, Paramount, Mirage Enterprises, Sand...","Germany, United States of America","French, English","chauffeur, sibling relationship, paris, france...",Sydney Pollack,"Samuel A. Taylor, Billy Wilder, Ernest Lehman,...",6.3,47256,"Harrison Ford, Julia Ormond, Greg Kinnear, Nan...",movie
7,8,112302,45325.0,45325,Tom and Huck,5.246,167,1995-12-22,23920048,97,...,"Walt Disney Pictures, Painted Fence Productions",United States of America,English,"based on novel or book, mississippi river, mal...",Peter Hewitt,"Mark Twain, Stephen Sommers, David Loughery",5.5,12450,"Jonathan Taylor Thomas, Brad Renfro, Eric Schw...",movie
8,9,114576,9091.0,9091,Sudden Death,6.000,638,1995-10-27,64350171,110,...,"SHATTERED PRODUCTIONS, Universal Pictures, Imp...",United States of America,English,"explosive, hostage, ice hockey, terrorism, vic...",Peter Hyams,"Karen Elise Baldwin, Gene Quintano",5.9,39484,"Jean-Claude Van Damme, Powers Boothe, Raymond ...",movie
9,10,113189,710.0,710,GoldenEye,6.880,3530,1995-11-16,352194034,130,...,"Eon Productions, United Artists","United Kingdom, United States of America","English, Russian, Spanish","computer virus, cuba, falsely accused, secret ...",Martin Campbell,"Ian Fleming, Michael France, Jeffrey Caine, Br...",7.2,285568,"Pierce Brosnan, Sean Bean, Izabella Scorupco, ...",movie


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8325 entries, 0 to 8324
Data columns (total 31 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   movieId               8325 non-null   int64  
 1   imdbId                8325 non-null   int64  
 2   tmdbId                8324 non-null   float64
 3   id                    8325 non-null   int64  
 4   title                 8325 non-null   object 
 5   vote_average          8325 non-null   float64
 6   vote_count            8325 non-null   int64  
 7   release_date          8285 non-null   object 
 8   revenue               8325 non-null   int64  
 9   runtime               8325 non-null   int64  
 10  adult                 8325 non-null   bool   
 11  backdrop_path         8089 non-null   object 
 12  budget                8325 non-null   int64  
 13  homepage              2289 non-null   object 
 14  original_language     8325 non-null   object 
 15  original_title       

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.duplicated(subset=["imdbId", "titleType"]).sum()

np.int64(77)

In [ ]:
print(df["imdbId"][:10])

0    114709
1    113497
2    113228
3    114885
4    113041
5    113277
6    114319
7    112302
8    114576
9    113189
Name: imdbId, dtype: int64


In [ ]:
# df["tconst"] = df["tconst"].astype("int32")

In [ ]:
df[df.duplicated(subset=["imdbId"], keep=False)].sort_values("imdbId")

,movieId,imdbId,tmdbId,id,title,vote_average,vote_count,release_date,revenue,runtime,...,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast,titleType
4883,26614,94791,8677.0,895795,The Bourne Identity,4.0,1,1988-05-08,0,185,...,NaN,United States of America,English,NaN,Roger Young,"Robert Ludlum, Carol Sobieski",6.8,5172,NaN,tvMiniSeries
4884,26614,94791,8677.0,895795,The Bourne Identity,4.0,1,1988-05-08,0,185,...,NaN,United States of America,English,NaN,Roger Young,"Robert Ludlum, Carol Sobieski",6.8,5172,"Richard Chamberlain, Jaclyn Smith, Donald Moff...",tvMiniSeries
4885,26614,94791,8677.0,1316934,The Bourne Identity,0.0,0,1988-05-08,0,185,...,NaN,NaN,English,NaN,Roger Young,"Robert Ludlum, Carol Sobieski",6.8,5172,NaN,tvMiniSeries
4886,26614,94791,8677.0,1316934,The Bourne Identity,0.0,0,1988-05-08,0,185,...,NaN,NaN,English,NaN,Roger Young,"Robert Ludlum, Carol Sobieski",6.8,5172,"Richard Chamberlain, Jaclyn Smith, Donald Moff...",tvMiniSeries
4893,26693,99864,2670.0,1329041,IT,8.0,1,1990-11-18,0,0,...,"Green/Epstein Productions, Lorimar Television,...",United States of America,English,"based on novel or book, supernatural, clown, f...",Tommy Lee Wallace,"Stephen King, Tommy Lee Wallace, Lawrence D. C...",6.8,152822,"Tim Curry, Harry Anderson, Annette O'Toole, De...",tvMiniSeries
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7616,130842,4475970,327029.0,1634524,power/rangers,0.0,0,NaN,0,14,...,NaN,NaN,NaN,NaN,Joseph Kahn,"Joseph Kahn, James Van Der Beek, Dutch Souther...",7.6,3877,NaN,short
7617,130842,4475970,327029.0,1634524,power/rangers,0.0,0,NaN,0,14,...,NaN,NaN,NaN,NaN,Joseph Kahn,"Joseph Kahn, James Van Der Beek, Dutch Souther...",7.6,3877,"Katee Sackhoff, James Van Der Beek, Russ Bain,...",short
7613,130842,4475970,327029.0,1529154,Power/Rangers,0.0,0,2015-01-01,0,14,...,NaN,United States of America,English,super heroes,Joseph Kahn,"Joseph Kahn, James Van Der Beek, Dutch Souther...",7.6,3877,"Katee Sackhoff, James Van Der Beek, Russ Bain,...",short
8215,175693,5950978,412103.0,1518115,State of Georgia Vs. Denver Fenton Allen,0.0,0,NaN,0,0,...,NaN,NaN,NaN,NaN,Erica Hayes,Fay Frankland,8.5,1092,"James Atkinson, Justin Roiland",tvShort


In [ ]:
df["title"] = df["title"].str.strip()

In [ ]:
df["merge_key"] = df["imdbId"].astype(str) + "_" + df["title"]

In [ ]:
def first_valid(series):
    valid = series.dropna()
    return valid.iloc[0] if len(valid) > 0 else None



In [ ]:
df_clean = df.groupby("merge_key").agg({

    # identity
    "movieId": "first",
    "imdbId": "first",
    "tmdbId": "first",
    "id": "first",
    "title": "first",

    # ratings
    "vote_average": "mean",
    "vote_count": "max",

    # financials
    "revenue": "max",
    "budget": "max",

    # time
    "release_date": first_valid,
    "runtime": "mean",

    # metadata
    "adult": "first",
    "backdrop_path": first_valid,
    "homepage": first_valid,
    "original_language": "first",
    "original_title": "first",
    "overview": first_valid,
    "popularity": "mean",
    "poster_path": first_valid,
    "tagline": first_valid

}).reset_index(drop=True)

In [ ]:
df_clean.duplicated(subset=["imdbId"]).sum()

np.int64(5)

In [ ]:
df_clean[df_clean.duplicated(subset=["imdbId"], keep=False)] \
.sort_values("imdbId")

,movieId,imdbId,tmdbId,id,title,vote_average,vote_count,revenue,budget,release_date,runtime,adult,backdrop_path,homepage,original_language,original_title,overview,popularity,poster_path,tagline
8244,26693,99864,2670.0,1329041,IT,4.0000,1,0,0,1990-11-18,1.5,False,/jkvyrnSODJqNsMs5m63zDADAZkl.jpg,None,en,IT,"In 1960, seven pre-teen outcasts fight an evil...",0.30000,/qCwDeVzXFCTSR7tmscExcQsyWGT.jpg,None
8245,26693,99864,2670.0,1390481,It,0.0000,0,0,0,None,192.0,False,None,None,en,It,"In 1960, seven outcast kids known as ""The Lose...",0.60000,/pqQSNba0oFWQnIBGtU7Tj8m27B2.jpg,The Master of Horror unleashes everything you ...
8246,26693,99864,2670.0,1590111,Stephen King's IT,8.0000,1,0,0,1990-11-18,192.0,False,/yZR0DgRWeTynfK6ZupKf74DW2D0.jpg,None,en,Stephen King's IT,"In 1960, seven pre-teen outcasts fight an evil...",0.14290,/2inEyJIenY4HIsKACntkL0INaue.jpg,Your every fear - all in one deadly enemy.
3379,119218,2280378,120605.0,1650031,The Punisher - Dirty Laundry,0.0000,0,0,0,2012-07-15,0.0,False,None,None,en,The Punisher - Dirty Laundry,None,0.02140,/htKZsqBgRRPy4dasAgnOpYhknN1.jpg,None
3380,119218,2280378,120605.0,120605,The Punisher: Dirty Laundry,3.5205,317,0,0,2012-07-16,5.5,False,/7mUdchgDrSZsw6T8sJgwZvNsWiP.jpg,https://www.rawstudios.com/video/dirty-laundry,en,The Punisher: Dirty Laundry,"In a bad neighborhood, on his way to a laundro...",4.42500,/5o5uPBWe0pYj7nHmE7V8yoNlLQb.jpg,None
5575,130842,4475970,327029.0,1634524,power/rangers,0.0000,0,0,0,None,14.0,False,None,None,en,power/rangers,A dark and gritty re-imagining of the classic ...,0.00000,None,None
5574,130842,4475970,327029.0,327029,Power/Rangers,3.4000,209,0,0,2015-02-24,14.0,False,/bZAvq63U9sfwIzNR0M35VCJ6yZB.jpg,https://vimeo.com/120401488,en,Power/Rangers,The Machine Empire defeats the Power Rangers i...,2.11366,/yZjqMc3PsnuoPKWDVcD0sggYXUF.jpg,The Machine Empire won. Welcome to the resista...
6304,175693,5950978,412103.0,1518115,State of Georgia Vs. Denver Fenton Allen,0.0000,0,0,0,None,0.0,False,None,None,en,State of Georgia Vs. Denver Fenton Allen,None,0.00000,None,None
6303,175693,5950978,412103.0,1365647,Rick and Morty: State of Georgia Vs. Denver Fe...,0.0000,0,0,0,2016-08-04,10.0,False,None,None,en,Rick and Morty: State of Georgia Vs. Denver Fe...,"A faithful, word-for-word recreation of one co...",1.40000,/A1gF7yVTcoBKwxKED23DOzKD5qI.jpg,None


In [ ]:
df["data_score"] = df.notna().sum(axis=1)

In [ ]:
df = df.sort_values(["imdbId", "data_score"], ascending=[True, False])

In [ ]:
df_clean = df.drop_duplicates(subset=["imdbId"], keep="first")

In [ ]:
df_clean[df_clean.duplicated(subset=["imdbId"], keep=False)] \
.sort_values("imdbId")

,movieId,imdbId,tmdbId,id,title,vote_average,vote_count,release_date,revenue,runtime,...,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast,titleType,merge_key,data_score


In [ ]:
# =========================
# 1. SAVE FILES
# =========================
df_clean.to_csv("advanced-imdb-train.csv", index=False)

# (if you have another dataframe, add it like this)
# df_other.to_csv("advanced-imdb-test.csv", index=False)

# =========================
# 2. CREATE DATASET FOLDER
# =========================
import os
os.makedirs("advanced-imdb", exist_ok=True)

os.replace("advanced-imdb-train.csv", "advanced-imdb/advanced-imdb-train.csv")
# os.replace("advanced-imdb-test.csv", "advanced-imdb/advanced-imdb-test.csv")  # optional

# =========================
# 3. METADATA (DO NOT CHANGE ID IF UPDATING)
# =========================
import json

metadata = {
    "title": "Advanced IMDb Dataset (Cleaned + Train Split)",
    "id": "sheikhmuneebahmed115/advanced-imdb",
    "licenses": [{"name": "CC0-1.0"}]
}

with open("advanced-imdb/dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

# =========================
# 4. INSTALL KAGGLE API
# =========================
!pip install -q kaggle

# =========================
# 5. UPLOAD KAGGLE TOKEN (manual step)
# =========================
from google.colab import files
files.upload()  # upload kaggle.json

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# =========================
# 6. UPLOAD DATASET (MULTI-FILE SUPPORTED)
# =========================
!kaggle datasets version -p advanced-imdb -m "added train file + updates"

Saving kaggle.json to kaggle.json
Starting upload for file advanced-imdb-train.csv
100% 7.87M/7.87M [00:00<00:00, 27.1MB/s]
Upload successful: advanced-imdb-train.csv (8MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/sheikhmuneebahmed115/advanced-imdb


In [ ]:
!mkdir advanced-imdb
!mv advanced-imdb.csv advanced-imdb/

In [ ]:

# ====== 3. write kaggle metadata ======
metadata = {
    "title": "Advanced IMDb Dataset",
    "id": "sheikhmuneebahmed115/advanced-imdb",
    "licenses": [{"name": "CC0-1.0"}]
}

import json
with open("advanced-imdb/dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

# ====== 4. install kaggle api ======
!pip install -q kaggle

# ====== 5. upload kaggle.json (you must upload manually in colab) ======
from google.colab import files
files.upload()  # upload kaggle.json here

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# ====== 6. create kaggle dataset ======
!kaggle datasets create -p advanced-imdb

Saving kaggle.json to kaggle.json
Starting upload for file advanced-imdb.csv
100% 155M/155M [00:02<00:00, 56.1MB/s]
Upload successful: advanced-imdb.csv (155MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/sheikhmuneebahmed115/advanced-imdb
